In [ ]:
%run ./phase2_common

In [ ]:
from datetime import date, datetime

dbutils.widgets.text("input_path", "", "Input JSON file or directory")
dbutils.widgets.dropdown("load_type", "full", ["full", "incremental"], "Load type")
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("source_extracted_at", "", "Snapshot time (ISO 8601 with timezone)")
dbutils.widgets.text("window_start", "", "Incremental window start (YYYY-MM-DD)")
dbutils.widgets.text("window_end", "", "Incremental window end (YYYY-MM-DD)")

names = ["input_path", "load_type", "batch_id", "source_extracted_at", "window_start", "window_end"]
p = {name: dbutils.widgets.get(name).strip() for name in names}

if not (p["input_path"] and p["batch_id"] and p["source_extracted_at"]):
    raise ValueError("input_path, batch_id and source_extracted_at are required")
if bool(p["window_start"]) != bool(p["window_end"]):
    raise ValueError("Provide both window dates, or leave both blank")
if p["load_type"] == "full" and p["window_start"]:
    raise ValueError("Full loads cannot have window dates")
for value in (p["window_start"], p["window_end"]):
    if value:
        date.fromisoformat(value)

snapshot_ts = datetime.fromisoformat(p["source_extracted_at"].replace("Z", "+00:00"))
if snapshot_ts.tzinfo is None:
    raise ValueError("source_extracted_at must include a timezone")

In [ ]:
wrapper_schema = (
    "load_type STRING, record_count BIGINT, window_start STRING, "
    "window_end STRING, records ARRAY<STRING>"
)
files = spark.read.option("multiLine", True).schema(wrapper_schema).json(p["input_path"])
if not files.limit(1).count():
    raise ValueError(f"No input files found at {p['input_path']}")

if files.filter(F.col("record_count") != F.size("records")).limit(1).count():
    raise ValueError("A file's record_count does not match its number of records")

records = (
    files.select(F.col("_metadata.file_path").alias("source_file"), F.explode("records").alias("raw_payload"))
    .withColumn("movie_id", F.from_json("raw_payload", "id BIGINT").getField("id"))
)
if records.filter(F.col("movie_id").isNull()).limit(1).count():
    raise ValueError("Some records have no movie ID")

conflicts = (
    records.groupBy("movie_id")
    .agg(F.countDistinct(F.sha2("raw_payload", 256)).alias("versions"))
    .filter("versions > 1")
)
if conflicts.limit(1).count():
    raise ValueError("A movie appears with conflicting payloads in this batch")

source = records.dropDuplicates(["movie_id"]).select(
    "movie_id",
    F.lit(p["load_type"]).alias("load_type"),
    F.lit(p["batch_id"]).alias("batch_id"),
    "source_file",
    F.lit(snapshot_ts).cast("timestamp").alias("source_extracted_at"),
    F.lit(p["window_start"] or None).cast("date").alias("window_start"),
    F.lit(p["window_end"] or None).cast("date").alias("window_end"),
    F.current_date().alias("ingestion_date"),
    F.current_timestamp().alias("load_timestamp"),
    "raw_payload",
)

In [ ]:
def load():
    counts = upsert(BRONZE, source, ["batch_id", "movie_id"], insert_only=True)
    stored = spark.table(BRONZE).filter(F.col("batch_id") == p["batch_id"]).count()
    if stored != counts["processed"]:
        raise RuntimeError(f"Expected {counts['processed']} Bronze rows for this batch, found {stored}")
    return counts


print(run_logged("Raw-to-Bronze", p["load_type"], p["input_path"], p["batch_id"], load))